# D181 - MySQL Query Optimization with Olist

This notebook covers three connected topics:

1. **Execution plans** - how MySQL intends to access tables, indexes, joins, grouping, and sorting.
2. **Query tuning** - how to rewrite queries and design indexes using evidence from plans.
3. **Best practices** - a repeatable workflow that improves performance without changing query meaning.

The examples use the Olist tables loaded by D16 and queried in D17. Run the notebook from top to bottom.

## Learning objectives

By the end, you should be able to:

- read important columns in traditional `EXPLAIN`;
- use `EXPLAIN FORMAT=JSON` and `EXPLAIN ANALYZE`;
- distinguish estimates from actual execution measurements;
- recognize full scans, index lookups, covering indexes, filesorts, and temporary tables;
- write sargable predicates;
- design single-column and composite indexes;
- apply the leftmost-prefix rule;
- improve joins, aggregations, top-N queries, CTEs, and subqueries;
- validate improvements rather than relying on guesses.

## 1. Connect to MySQL

The source database is `olist_import_lab`. The notebook also creates `query_optimization_lab` and a disposable copy named `olist_items_tuning_lab`. This keeps index experiments away from the D16/D17 source tables.

In [ ]:
import os
import time
import mysql.connector
from mysql.connector import Error

MYSQL_CONFIG = {
    'host': os.environ.get('MYSQL_HOSTNAME', '127.0.0.1'),
    'port': int(os.environ.get('MYSQL_PORT', '3306')),
    'user': os.environ.get('MYSQL_USERNAME', 'root'),
    'password': os.environ.get('MYSQL_PASSWORD', 'root'),
}

server = mysql.connector.connect(**MYSQL_CONFIG)
cursor = server.cursor()
cursor.execute('CREATE DATABASE IF NOT EXISTS query_optimization_lab')
cursor.close()
server.close()

connection = mysql.connector.connect(**MYSQL_CONFIG, database='query_optimization_lab')
print('Connected:', connection.is_connected())
print('MySQL version:', connection.get_server_info())

## 2. Reliable D14/D16 query helpers

`execute_sql` prints complete results, commits DDL/DML, rolls back errors, and returns rows or an affected-row count. `print_rows` keeps result tables readable. A separate `time_sql` helper is included, but elapsed time alone is noisy; plans and repeated measurements are more trustworthy.

In [ ]:
def print_rows(columns, rows):
    if not rows:
        print('No rows returned.')
        return
    text_rows = [['NULL' if value is None else str(value) for value in row]
                 for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    print(' | '.join(str(column).ljust(width)
                     for column, width in zip(columns, widths)))
    print('-+-'.join('-' * width for width in widths))
    for row in text_rows:
        print(' | '.join(value.ljust(width)
                         for value, width in zip(row, widths)))


def execute_sql(sql, params=None):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if cursor.with_rows:
            columns = [item[0] for item in cursor.description]
            rows = cursor.fetchall()
            print_rows(columns, rows)
            return rows
        affected = cursor.rowcount
        connection.commit()
        print(f'Statement completed. Affected rows: {affected:,}')
        return affected
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()


def time_sql(sql, params=None, repeats=3):
    durations = []
    row_count = 0
    for _ in range(repeats):
        cursor = connection.cursor()
        started = time.perf_counter()
        cursor.execute(sql, params or ())
        rows = cursor.fetchall()
        durations.append((time.perf_counter() - started) * 1000)
        row_count = len(rows)
        cursor.close()
    print(f'Rows: {row_count}; milliseconds: {durations}; best: {min(durations):.3f}')
    return durations

## 3. Confirm the Olist source

`olist_order_items` contains one row per item in an order. Its composite primary key is `(order_id, order_item_id)`. Existing source indexes are left unchanged.

In [ ]:
execute_sql("""
SELECT COUNT(*) AS item_rows,
       COUNT(DISTINCT order_id) AS orders,
       COUNT(DISTINCT seller_id) AS sellers,
       MIN(shipping_limit_date) AS first_shipping_limit,
       MAX(shipping_limit_date) AS last_shipping_limit
FROM olist_import_lab.olist_order_items
""")

execute_sql('SHOW INDEX FROM olist_import_lab.olist_order_items')

## 4. Create one isolated tuning table

`CREATE TABLE ... LIKE` copies columns and current indexes. For a clean before/after experiment, the copied secondary indexes are removed while the primary key is retained. The source data is copied once. Re-running this cell resets the lab.

In [ ]:
execute_sql('DROP TABLE IF EXISTS olist_items_tuning_lab')
execute_sql("""
CREATE TABLE olist_items_tuning_lab
LIKE olist_import_lab.olist_order_items
""")

# Remove copied secondary indexes; retain the composite primary key.
for index_name in ['idx_item_product', 'idx_item_seller']:
    try:
        execute_sql(f'ALTER TABLE olist_items_tuning_lab DROP INDEX {index_name}')
    except Error as exc:
        print(f'{index_name}: {exc.msg} (safe to ignore if the source used another name)')

execute_sql("""
INSERT INTO olist_items_tuning_lab
SELECT * FROM olist_import_lab.olist_order_items
""")
execute_sql('ANALYZE TABLE olist_items_tuning_lab')
execute_sql('SHOW INDEX FROM olist_items_tuning_lab')

# Part A - Execution Plans

## 5. What `EXPLAIN` does

`EXPLAIN SELECT ...` asks the optimizer for its chosen plan without running the `SELECT`. Important traditional columns include:

- `table`: table or derived result being accessed;
- `partitions`: selected partitions, when relevant;
- `type`: access method;
- `possible_keys`: indexes that might help;
- `key`: chosen index;
- `key_len`: how much of the index is used;
- `ref`: value or column used for lookup;
- `rows`: estimated rows examined;
- `filtered`: estimated percentage surviving the condition;
- `Extra`: additional work such as `Using index`, `Using where`, `Using temporary`, or `Using filesort`.

### Access types: generally better to worse

`system` / `const` -> `eq_ref` -> `ref` -> `range` -> `index` -> `ALL`

This is a useful warning scale, not an absolute score. `ALL` may be correct for a tiny table or when most rows are needed. `index` means a full index scan, which is not the same as a selective index lookup. Judge the complete plan and actual work.

In [ ]:
execute_sql("""
EXPLAIN
SELECT order_id, order_item_id, seller_id, price
FROM olist_items_tuning_lab
WHERE seller_id = '6560211a19b47992c3666cc44a7e94c0'
""")

The lab initially has no seller index, so the expected access type is `ALL`. `possible_keys` and `key` should be `NULL`, and estimated rows should be close to the table size.

## 6. `EXPLAIN FORMAT=JSON`

JSON plans expose nested query blocks, cost estimates, used columns, attached conditions, chosen keys, and grouping/sorting operations. They are easier for tools to parse than the traditional table. Costs are optimizer units, not milliseconds.

In [ ]:
execute_sql("""
EXPLAIN FORMAT=JSON
SELECT seller_id, COUNT(*) AS items, SUM(price) AS item_value
FROM olist_items_tuning_lab
WHERE shipping_limit_date >= '2018-01-01'
GROUP BY seller_id
ORDER BY item_value DESC
LIMIT 10
""")

## 7. `EXPLAIN ANALYZE`: estimates versus reality

`EXPLAIN ANALYZE` actually executes the statement and reports iterator timing, estimated versus actual rows, and loop counts. It is the strongest built-in tool for detecting bad cardinality estimates. Use it carefully: the query really runs, so do not casually analyze expensive or modifying production statements.

In [ ]:
execute_sql("""
EXPLAIN ANALYZE
SELECT seller_id, COUNT(*) AS items, SUM(price) AS item_value
FROM olist_items_tuning_lab
WHERE shipping_limit_date >= '2018-01-01'
GROUP BY seller_id
ORDER BY item_value DESC
LIMIT 10
""")

# Part B - Query Tuning

## 8. Add an index supported by the workload

The equality predicate filters by seller and returns a few columns. A composite index beginning with `seller_id` supports the lookup. Adding `price` makes seller-value aggregation more likely to use a covering index. Primary-key columns are stored with InnoDB secondary-index entries, so `order_id` and `order_item_id` can also be available from the index.

In [ ]:
execute_sql("""
ALTER TABLE olist_items_tuning_lab
ADD INDEX idx_seller_price (seller_id, price)
""")
execute_sql('ANALYZE TABLE olist_items_tuning_lab')

execute_sql("""
EXPLAIN
SELECT order_id, order_item_id, seller_id, price
FROM olist_items_tuning_lab
WHERE seller_id = '6560211a19b47992c3666cc44a7e94c0'
""")

Look for `ref` access, `idx_seller_price` in `key`, a much smaller row estimate, and possibly `Using index`. A covering index avoids extra table-row lookups, but wider indexes cost storage and make writes more expensive.

## 9. Sargability: do not hide indexed columns

A predicate is sargable when MySQL can turn it into an index search range. Applying a function to the indexed column often prevents that.

Poor: `YEAR(shipping_limit_date) = 2018`

Better: `shipping_limit_date >= '2018-01-01' AND shipping_limit_date < '2019-01-01'`

The half-open interval is safe for both `DATE` and `DATETIME`.

In [ ]:
execute_sql('ALTER TABLE olist_items_tuning_lab ADD INDEX idx_shipping_date (shipping_limit_date)')
execute_sql('ANALYZE TABLE olist_items_tuning_lab')

print('Non-sargable plan')
execute_sql("""
EXPLAIN SELECT COUNT(*) FROM olist_items_tuning_lab
WHERE YEAR(shipping_limit_date) = 2018
""")

print('Sargable plan')
execute_sql("""
EXPLAIN SELECT COUNT(*) FROM olist_items_tuning_lab
WHERE shipping_limit_date >= '2018-01-01'
  AND shipping_limit_date <  '2019-01-01'
""")

## 10. Composite indexes and the leftmost-prefix rule

For `(seller_id, shipping_limit_date, price)`, MySQL can efficiently use prefixes beginning with `seller_id`:

- `seller_id = ...`;
- `seller_id = ... AND shipping_limit_date >= ...`;
- potentially seller plus date plus price, subject to range rules.

A predicate only on `shipping_limit_date` cannot normally seek through this index because the leftmost column is missing. Column order should follow actual equality, range, grouping, and ordering needs—not a universal selectivity rule.

In [ ]:
execute_sql("""
ALTER TABLE olist_items_tuning_lab
ADD INDEX idx_seller_date_price (seller_id, shipping_limit_date, price)
""")
execute_sql('ANALYZE TABLE olist_items_tuning_lab')

execute_sql("""
EXPLAIN SELECT seller_id, shipping_limit_date, price
FROM olist_items_tuning_lab
WHERE seller_id = '6560211a19b47992c3666cc44a7e94c0'
  AND shipping_limit_date >= '2018-01-01'
ORDER BY shipping_limit_date
""")

### Equality columns before a range column

A common composite-index pattern is equality filters first, followed by one range or ordering column. Once a range condition is used, later columns often cannot further narrow the index search, although they may still help covering or filtering. Always inspect `key_len`, JSON plan details, and actual rows.

## 11. Tune joins by preserving indexed keys and correct grain

Olist orders and items join on `order_id`, which is indexed in both source tables. Avoid wrapping join keys in functions or converting types. Also aggregate at the required business grain before joining detail tables, both for correctness and less work.

In [ ]:
execute_sql("""
EXPLAIN
SELECT o.order_status, COUNT(*) AS item_rows, SUM(i.price) AS item_value
FROM olist_import_lab.olist_orders AS o
JOIN olist_import_lab.olist_order_items AS i
  ON i.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY o.order_status
""")

In a good join plan, MySQL filters one side and uses `eq_ref` or `ref` lookups on the other. The best join order depends on cardinalities and available indexes; SQL text order does not force optimizer join order for ordinary inner joins.

## 12. `EXISTS` for existence tests

If the requirement is only to know whether a related row exists, `EXISTS` communicates that intent and avoids multiplying parent rows. MySQL may transform it into an efficient semijoin. `JOIN` is appropriate when related columns or multiplicity are required.

In [ ]:
execute_sql("""
EXPLAIN
SELECT o.order_id, o.order_purchase_timestamp
FROM olist_import_lab.olist_orders AS o
WHERE EXISTS (
    SELECT 1
    FROM olist_import_lab.olist_order_items AS i
    WHERE i.order_id = o.order_id
      AND i.price >= 1000
)
""")

## 13. Top-N queries and sorting

`ORDER BY ... LIMIT 10` is fast only when MySQL can avoid or cheaply perform the sort. `Using filesort` means MySQL performs an extra sort algorithm; it does not necessarily mean disk I/O. An index can provide order when its leading columns and directions match the filter/order pattern. Aggregated values such as `SUM(price)` usually must be calculated before sorting.

In [ ]:
execute_sql("""
EXPLAIN ANALYZE
SELECT seller_id, SUM(price) AS item_value
FROM olist_items_tuning_lab
GROUP BY seller_id
ORDER BY item_value DESC
LIMIT 10
""")

The seller-leading covering index can reduce table reads during grouping, but it cannot directly store order by the runtime aggregate. For frequently requested leaderboards, consider a maintained summary table rather than repeatedly aggregating a large fact table.

## 14. CTEs, derived tables, and materialization

CTEs improve structure but are not inherently faster. MySQL may merge a CTE into the outer query or materialize it as an internal temporary result. Reusing a costly CTE, grouping inside it, or certain constructs can encourage materialization. Inspect the plan rather than assuming.

In [ ]:
execute_sql("""
EXPLAIN FORMAT=JSON
WITH seller_totals AS (
    SELECT seller_id, COUNT(*) AS items, SUM(price) AS item_value
    FROM olist_items_tuning_lab
    GROUP BY seller_id
)
SELECT seller_id, items, item_value
FROM seller_totals
WHERE item_value >= 100000
ORDER BY item_value DESC
""")

## 15. Window functions require sorting and correct grain

Window functions often sort each partition. Prepare one row per business entity before ranking. Ranking raw item rows for a seller leaderboard would be incorrect and would sort more rows.

In [ ]:
execute_sql("""
EXPLAIN ANALYZE
WITH seller_totals AS (
    SELECT seller_id, SUM(price) AS item_value
    FROM olist_items_tuning_lab
    GROUP BY seller_id
), ranked AS (
    SELECT seller_id, item_value,
           DENSE_RANK() OVER (ORDER BY item_value DESC) AS seller_rank
    FROM seller_totals
)
SELECT seller_id, item_value, seller_rank
FROM ranked
WHERE seller_rank <= 10
ORDER BY seller_rank, seller_id
""")

## 16. Avoid unnecessary work

- Select only required columns instead of `SELECT *`.
- Filter early when doing so preserves meaning.
- Avoid unnecessary `DISTINCT`; it may hide an incorrect join and adds deduplication work.
- Use `UNION ALL` when duplicate removal is not required.
- Avoid leading-wildcard searches such as `LIKE '%text'` on ordinary B-tree indexes.
- Do not use functions on filter/join columns when an equivalent range predicate exists.
- Do not request an order unless the result truly needs one.
- Use realistic limits, but remember `LIMIT` does not guarantee that little data is scanned.

## 17. Statistics and cardinality estimates

The optimizer chooses plans from statistics. Run `ANALYZE TABLE` after major data changes when estimates are stale. Histograms can help skewed, nonindexed columns, but should be created only when evidence shows poor estimates. Compare estimated rows with actual rows in `EXPLAIN ANALYZE`.

In [ ]:
execute_sql('ANALYZE TABLE olist_items_tuning_lab')
execute_sql("""
SELECT table_name, table_rows, data_length, index_length
FROM information_schema.tables
WHERE table_schema = DATABASE()
  AND table_name = 'olist_items_tuning_lab'
""")
execute_sql("""
SELECT index_name, seq_in_index, column_name, cardinality
FROM information_schema.statistics
WHERE table_schema = DATABASE()
  AND table_name = 'olist_items_tuning_lab'
ORDER BY index_name, seq_in_index
""")

# Part C - Query Optimization Best Practices

## 18. A repeatable tuning workflow

1. **Define correctness and the required grain.** Record expected row counts and totals.
2. **Capture a baseline.** Save SQL, parameters, plan, runtime distribution, rows examined, and environment.
3. **Find the expensive iterator.** Use `EXPLAIN ANALYZE`, not intuition alone.
4. **Check cardinality estimates.** Large estimate/actual differences can produce poor join orders or access choices.
5. **Reduce work.** Improve predicates, joins, selected columns, aggregation grain, and sorting.
6. **Design the smallest useful index.** Consider equality, range, join, group/order, and covering columns.
7. **Change one thing at a time.** Re-run the same plan and workload.
8. **Measure trade-offs.** Indexes consume storage and slow inserts, updates, and deletes.
9. **Test representative parameters.** Selectivity can vary dramatically.
10. **Monitor after deployment.** Data volume and distribution change over time.

## 19. Index design checklist

Before adding an index, ask:

- Which frequent query will use it?
- Are leading columns constrained by equality?
- Where is the first range condition?
- Does column order support the required ordering or grouping?
- Is covering worth the additional width?
- Does a current index already cover the same left prefix?
- How selective are the values in real data?
- What is the write and storage cost?
- Can the index be removed safely if it is unused?

Avoid indexing every column independently. Several isolated indexes are not equivalent to one well-designed composite index, although MySQL can sometimes use index merge.

## 20. Production cautions

- Do not run `EXPLAIN ANALYZE` on an expensive production query without understanding that it executes.
- Test DDL duration and locking behavior before adding large indexes.
- Prefer online DDL capabilities where supported, but verify the actual algorithm and lock.
- Query cache behavior, buffer-pool warmth, concurrency, and network transfer can distort timing.
- Avoid optimizer hints until schema, statistics, and SQL design have been investigated. Hints can become stale as data changes.
- Never tune by changing semantics: dropped rows, duplicated money, or nondeterministic top-N results are not optimizations.

## 21. Practice tasks

1. Find the plan for filtering `product_id` before and after adding a product-leading index.
2. Compare `DATE(shipping_limit_date) = '2018-01-01'` with a half-open datetime range.
3. Design an index for seller, date range, and price output; verify `key_len`.
4. Compare an existence `JOIN DISTINCT` with `EXISTS`.
5. Use `EXPLAIN ANALYZE` to compare estimated and actual seller rows.
6. Determine whether the seller leaderboard uses a temporary table or filesort.
7. Remove one redundant lab index and confirm which plan changes.

## 22. Optional cleanup

The tuning table is intentionally retained so you can repeat exercises. Run the next statement only when the lab copy is no longer needed. It does not touch the Olist source schema.

In [ ]:
# Uncomment to remove the disposable tuning table.
# execute_sql('DROP TABLE IF EXISTS olist_items_tuning_lab')

## 23. Close the connection

In [ ]:
if connection.is_connected():
    connection.close()
print('MySQL connection closed.')